In [ ]:
# %%
def main(datasources, start_date, end_date):
    """
    因子：残差学习XGBoost + 订单流增强 (Residual ML + OrderFlow Proxy)
    
    核心改进：在策略2残差学习框架上，新增6个订单流代理特征：
    - buy_pressure / sell_pressure: 日内买卖方向成交量占比
    - pressure_imbalance: 买卖压力差
    - large_order_proxy: 单笔均额Z-Score(大单活跃度代理)
    - flow_toxicity: 订单流毒性(知情交易概率代理)
    - order_flow_momentum: 5日压力差动量
    
    参数/返回值格式与原版完全一致，可直接替换评测
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb
    import structlog
    from sklearn.linear_model import LinearRegression

    logger = structlog.get_logger()

    TRAIN_START = '2022-01-01 00:00:00'
    TRAIN_END   = '2023-12-31 23:59:59'
    LOOKBACK_DAYS = 120

    PRICE_FEATURES = [
        'ret_1', 'ret_5', 'ret_20', 'ret_60',
        'vol_5', 'vol_20', 'range_1',
        'volume_z', 'amount_z', 'volume_trend',
        'close_position', 'vwap_deviation', 'signed_vol_ratio',
    ]

    # 【新增】订单流代理特征
    ORDERFLOW_FEATURES = [
        'buy_pressure', 'sell_pressure', 'pressure_imbalance',
        'large_order_proxy', 'flow_toxicity', 'order_flow_momentum',
    ]

    FIN_RAW_COLS = [
        'operating_revenue', 'net_profit_to_parent_shareholders',
        'total_assets', 'total_equity_to_parent_shareholders',
    ]
    FIN_RATIO_COLS = ['roe', 'asset_turnover', 'leverage', 'profit_margin']

    EXPOSURE_COLS = [
        'SIZE', 'BETA', 'MOMENTUM', 'RESVOL', 'SIZENL',
        'BTOP', 'LIQUIDTY', 'EARNYILD', 'GROWTH', 'LEVERAGE',
    ]

    ALL_PRICE_FIN_FEATURES = PRICE_FEATURES + FIN_RATIO_COLS

    RESIDUAL_CONTROL_COLS = EXPOSURE_COLS.copy()

    CANDIDATE_INDUSTRY_COLS = [
        'AGRIFOREST', 'MINING', 'CHEM', 'IRONSTEEL', 'NONFERMETAL',
        'ELECTRONICS', 'AUTO', 'HOUSEAPP', 'FOODBEVER', 'TEXTILE',
        'LIGHTINDUS', 'HEALTH', 'UTILITIES', 'TRANSPORTATION',
        'REALESTATE', 'COMMETRADE', 'LEISERVICE', 'BANK',
        'NONBANKFINAN', 'CONGLOMERATES', 'CONMAT', 'BUILDDECO',
        'ELECEQP', 'MACHIEQUIP', 'AERODEF', 'COMPUTER', 'MEDIA',
        'TELECOM', 'COAL', 'PETRO', 'ENVP', 'BEAUTY',
    ]

    def build_features(financial_table, bar1m_table, sd, ed):
        """构建量价+订单流+财务特征矩阵（含原始label）"""
        t0 = time.time()
        logger.info("build_features 开始", start=str(sd), end=str(ed))

        price_query_start = (
            pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        ).strftime('%Y-%m-%d %H:%M:%S')

        # SQL: 量价特征 + 订单流代理特征一次性计算
        price_sql = f"""
        WITH cte_bars AS (
            SELECT date, instrument, date::DATE AS trading_day,
                   open, high, low, close, volume, amount,
                   lag(close, 1) OVER (
                       PARTITION BY instrument, date::DATE ORDER BY date
                   ) AS prev_close
            FROM {bar1m_table}
            WHERE volume > 0 AND close > 0 AND open > 0 AND high > 0 AND low > 0
        ),
        daily_raw AS (
            SELECT CAST(trading_day AS DATETIME) AS date, instrument,
                   arg_min(open, date) AS open, max(high) AS high,
                   min(low) AS low, arg_max(close, date) AS close,
                   sum(volume)::DOUBLE AS volume, sum(amount)::DOUBLE AS amount,
                   sum(amount) / NULLIF(sum(volume), 0) AS vwap,
                   (arg_max(close, date) - min(low))
                       / NULLIF(max(high) - min(low), 0) AS close_position,
                   sum(CASE WHEN close > open THEN volume
                            WHEN close < open THEN -volume ELSE 0 END) AS signed_vol,
                   sum(abs(close / NULLIF(prev_close, 0) - 1)) AS path_length,
                   count(*) AS bar_count,
                   -- 【订单流】按K线涨跌方向拆分买卖成交量
                   sum(CASE WHEN close >= open 
                       THEN (close - low) / NULLIF(high - low, 0) * volume 
                       ELSE 0 END) AS raw_buy_vol,
                   sum(CASE WHEN close < open 
                       THEN (high - close) / NULLIF(high - low, 0) * volume 
                       ELSE 0 END) AS raw_sell_vol,
                   avg(amount / NULLIF(volume, 0)) AS avg_ticket_size
            FROM cte_bars GROUP BY trading_day, instrument
        ),
        daily_with_lag AS (
            SELECT date, instrument, open, high, low, close, volume, amount,
                   vwap, close_position, signed_vol, path_length, bar_count,
                   raw_buy_vol, raw_sell_vol, avg_ticket_size,
                   close / NULLIF(lag(close, 1) OVER w, 0) - 1 AS ret_1,
                   close / NULLIF(lag(close, 5) OVER w, 0) - 1 AS ret_5,
                   close / NULLIF(lag(close, 20) OVER w, 0) - 1 AS ret_20,
                   close / NULLIF(lag(close, 60) OVER w, 0) - 1 AS ret_60,
                   lead(close, 1) OVER w / NULLIF(close, 0) - 1 AS label,
                   (high - low) / NULLIF(close, 0) AS range_1,
                   ln(volume + 1) AS log_volume, ln(amount + 1) AS log_amount
            FROM daily_raw WINDOW w AS (PARTITION BY instrument ORDER BY date)
        ),
        daily_features AS (
            SELECT *,
                   nanstd(ret_1) OVER (ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS vol_5,
                   nanstd(ret_1) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol_20,
                   (log_volume - avg(log_volume) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW))
                   / NULLIF(nanstd(log_volume) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 0) AS volume_z,
                   (log_amount - avg(log_amount) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW))
                   / NULLIF(nanstd(log_amount) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 0) AS amount_z,
                   avg(volume) OVER (ROWS BETWEEN 4 PRECEDING AND CURRENT ROW)
                   / NULLIF(avg(volume) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 0) - 1 AS volume_trend,
                   (close - vwap) / NULLIF(vwap, 0) AS vwap_deviation,
                   signed_vol / NULLIF(volume, 0) AS signed_vol_ratio,
                   count(*) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS rolling_cnt_20,
                   -- 【订单流】标准化买卖压力
                   raw_buy_vol / NULLIF(volume, 0) AS buy_pressure,
                   raw_sell_vol / NULLIF(volume, 0) AS sell_pressure,
                   -- 【订单流】大单代理: 单笔均额的20日Z-Score
                   (avg_ticket_size - avg(avg_ticket_size) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW))
                   / NULLIF(nanstd(avg_ticket_size) OVER (ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 0) AS large_order_proxy
            FROM daily_with_lag
        )
        SELECT date, instrument,
               ret_1, ret_5, ret_20, ret_60, vol_5, vol_20, range_1,
               volume_z, amount_z, volume_trend, close_position,
               vwap_deviation, signed_vol_ratio, label,
               buy_pressure, sell_pressure, large_order_proxy
        FROM daily_features
        WHERE bar_count >= 30 AND rolling_cnt_20 >= 15
        ORDER BY instrument, date
        """

        price_df = dai.query(price_sql, filters={'date': [price_query_start, ed]}, compression=True).df()
        if price_df.empty:
            logger.warning("量价特征查询为空")
            return pd.DataFrame()

        price_df['date'] = pd.to_datetime(price_df['date'])
        price_df = price_df.sort_values(['instrument', 'date']).reset_index(drop=True)

        # Pandas侧计算衍生订单流特征（轻量级，仅3列运算）
        price_df['pressure_imbalance'] = price_df['buy_pressure'] - price_df['sell_pressure']
        denom = (price_df['buy_pressure'] + price_df['sell_pressure']).replace(0, np.nan)
        price_df['flow_toxicity'] = price_df['pressure_imbalance'].abs() / denom
        price_df['order_flow_momentum'] = (
            price_df.groupby('instrument')['pressure_imbalance']
            .transform(lambda x: x.rolling(5, min_periods=1).mean())
        )

        # 财务特征（与原版相同）
        fin_start = pd.to_datetime(sd) - pd.Timedelta(days=365)
        fin_select = ', '.join(FIN_RAW_COLS)
        fin_sql = f"SELECT date, instrument, {fin_select} FROM {financial_table} WHERE category='lf' AND shift=0"
        fin = dai.query(fin_sql, filters={'date': [fin_start, ed]}).df()

        if not fin.empty:
            fin['date'] = pd.to_datetime(fin['date'])
            natural_dates = pd.date_range(start=fin_start, end=pd.to_datetime(ed))
            def _ffill_group(grp):
                grp = grp.set_index('date').reindex(natural_dates)
                grp['instrument'] = grp['instrument'].ffill().bfill()
                for c in FIN_RAW_COLS:
                    grp[c] = grp[c].ffill()
                return grp
            fin = (fin.groupby('instrument', group_keys=False).apply(_ffill_group)
                   .reset_index().rename(columns={'index': 'date'}).dropna(subset=['instrument']))
            fin['date'] = pd.to_datetime(fin['date'])
            fin['roe'] = fin['net_profit_to_parent_shareholders'] / fin['total_equity_to_parent_shareholders'].replace(0, np.nan)
            fin['asset_turnover'] = fin['operating_revenue'] / fin['total_assets'].replace(0, np.nan)
            fin['leverage'] = fin['total_assets'] / fin['total_equity_to_parent_shareholders'].replace(0, np.nan)
            fin['profit_margin'] = fin['net_profit_to_parent_shareholders'] / fin['operating_revenue'].replace(0, np.nan)
            for c in FIN_RATIO_COLS:
                lo, hi = fin[c].quantile(0.01), fin[c].quantile(0.99)
                fin[c] = fin[c].clip(lo, hi)

        # 合并量价+订单流+财务
        all_price_of_cols = PRICE_FEATURES + ORDERFLOW_FEATURES
        for col in all_price_of_cols:
            if col in price_df.columns:
                price_df[col] = pd.to_numeric(price_df[col], errors='coerce')

        if not fin.empty:
            fin_cols_to_merge = ['date', 'instrument'] + FIN_RAW_COLS + FIN_RATIO_COLS
            df = pd.merge(price_df, fin[fin_cols_to_merge], how='left', on=['date', 'instrument'])
            for c in FIN_RAW_COLS + FIN_RATIO_COLS:
                df[c] = df.groupby('instrument')[c].ffill()
        else:
            df = price_df.copy()
            for c in FIN_RAW_COLS + FIN_RATIO_COLS:
                df[c] = np.nan

        all_feature_cols = PRICE_FEATURES + FIN_RATIO_COLS + ORDERFLOW_FEATURES
        for col in all_feature_cols:
            if col in df.columns:
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)

        df = df[(df['date'] >= pd.to_datetime(sd)) & (df['date'] <= pd.to_datetime(ed))]
        df = df.reset_index(drop=True)

        logger.info("build_features 结束", rows=len(df), elapsed=round(time.time() - t0, 2))
        return df

    def compute_residual_label(df, control_cols):
        """按日期横截面OLS回归，将label替换为残差"""
        t0 = time.time()
        available_controls = [c for c in control_cols if c in df.columns]

        if len(available_controls) == 0:
            logger.warning("无可用控制变量，退化为原始label")
            return df

        logger.info("计算残差标签", control_vars=len(available_controls))
        residual_labels = []

        for date_val, group in df.groupby('date'):
            y = group['label'].values
            X = group[available_controls].values
            valid_mask = ~(np.isnan(y) | np.any(np.isnan(X), axis=1))
            result = np.full(len(group), np.nan)

            if valid_mask.sum() > len(available_controls) + 1:
                reg = LinearRegression(fit_intercept=True)
                reg.fit(X[valid_mask], y[valid_mask])
                residuals = y[valid_mask] - reg.predict(X[valid_mask])
                result[valid_mask] = residuals
            else:
                valid_y = y[valid_mask]
                if len(valid_y) > 0:
                    result[valid_mask] = valid_y - valid_y.mean()

            residual_labels.append(result)

        df['label'] = np.concatenate(residual_labels)
        logger.info("残差标签完成", elapsed=round(time.time() - t0, 2),
                    valid_ratio=df['label'].notna().mean())
        return df

    # ============================================================
    # Step 1: 构建训练集
    # ============================================================
    logger.info("=" * 60)
    logger.info("构建训练集", train_start=TRAIN_START, train_end=TRAIN_END)
    logger.info("=" * 60)

    train_df = build_features(
        'bigalpha_2026_financial',
        'bigalpha_2026_stock_bar1m',
        TRAIN_START, TRAIN_END,
    )

    if train_df.empty:
        logger.error("训练集为空")
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    train_df['label'] = train_df['label'].replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=['label'])

    # 加载exposure
    exposure_features = []
    industry_cols = []
    use_exposure = False
    all_control_cols = []

    try:
        exposure_table = datasources.get('exposure', 'bigalpha_2026_exposure')
        exp_sample = dai.query(f"SELECT * FROM {exposure_table} LIMIT 1").df()

        if not exp_sample.empty:
            industry_cols = [c for c in CANDIDATE_INDUSTRY_COLS if c in exp_sample.columns]
            available_exposure = [c for c in EXPOSURE_COLS if c in exp_sample.columns]

            if available_exposure:
                exposure_features = available_exposure + industry_cols
                use_exposure = True
                all_control_cols = available_exposure + industry_cols

                logger.info("exposure可用", barra=len(available_exposure),
                            industries=len(industry_cols))

                exp_query_start = (
                    pd.to_datetime(TRAIN_START) - pd.Timedelta(days=10)
                ).strftime('%Y-%m-%d %H:%M:%S')
                exp_select = ', '.join(exposure_features)
                exp_train = dai.query(
                    f"SELECT date, instrument, {exp_select} FROM {exposure_table}",
                    filters={'date': [exp_query_start, TRAIN_END]}, compression=True,
                ).df()

                if not exp_train.empty:
                    exp_train['date'] = pd.to_datetime(exp_train['date'])
                    for c in exposure_features:
                        exp_train[c] = pd.to_numeric(exp_train[c], errors='coerce')
                    train_df = pd.merge(train_df, exp_train, how='left', on=['date', 'instrument'])
                    for c in exposure_features:
                        train_df[c] = train_df.groupby('instrument')[c].ffill()
    except Exception as e:
        logger.info("exposure不可用", reason=str(e)[:100])

    # 残差学习
    if use_exposure and len(all_control_cols) > 0:
        logger.info("【残差学习】剔除风格+行业线性影响...")
        train_df = compute_residual_label(train_df, all_control_cols)
    else:
        logger.warning("无控制变量，使用原始label")

    train_df['label'] = train_df['label'].replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=['label'])

    # 完整特征列表 = 量价 + 财务 + 订单流 + exposure
    all_features = list(PRICE_FEATURES + FIN_RATIO_COLS + ORDERFLOW_FEATURES)
    if use_exposure:
        all_features += exposure_features

    for c in all_features:
        if c not in train_df.columns:
            train_df[c] = np.nan

    # XGBoost训练
    t_fit = time.time()
    n_valid = max(int(len(train_df) * 0.1), 1000)
    train_split = train_df.iloc[:-n_valid].copy()
    valid_split = train_df.iloc[-n_valid:].copy()

    train_medians = {}
    for col in all_features:
        train_split[col] = pd.to_numeric(train_split[col], errors='coerce')
        valid_split[col] = pd.to_numeric(valid_split[col], errors='coerce')
        med = train_split[col].median()
        train_medians[col] = med if not pd.isna(med) else 0.0
        train_split[col] = train_split[col].fillna(train_medians[col])
        valid_split[col] = valid_split[col].fillna(train_medians[col])

    logger.info("训练XGBoost(目标=残差)", train=len(train_split),
                valid=len(valid_split), features=len(all_features))

    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=15,
        reg_lambda=3.0, objective='reg:squarederror',
        tree_method='hist', n_jobs=-1, random_state=42,
        early_stopping_rounds=20,
    )
    model.fit(
        train_split[all_features], train_split['label'],
        eval_set=[(valid_split[all_features], valid_split['label'])],
        verbose=False,
    )

    logger.info("训练完成", elapsed=round(time.time() - t_fit, 2),
                best_iter=model.get_booster().best_iteration,
                best_score=round(model.get_booster().best_score, 6))

    # Top特征重要性（验证订单流特征是否被模型采纳）
    imp = model.feature_importances_
    top_idx = np.argsort(imp)[-10:][::-1]
    logger.info("Top10特征", top=[(all_features[i], round(imp[i], 4)) for i in top_idx])

    # ============================================================
    # Step 2: 测试集预测
    # ============================================================
    logger.info("=" * 60)
    logger.info("测试集预测", test_start=start_date, test_end=end_date)
    logger.info("=" * 60)

    bar1m_table = datasources['bar1m']
    financial_table = datasources['financial']
    test_df = build_features(financial_table, bar1m_table, start_date, end_date)

    if test_df.empty:
        logger.error("测试集为空")
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    for c in all_features:
        if c not in test_df.columns:
            test_df[c] = np.nan
        test_df[c] = pd.to_numeric(test_df[c], errors='coerce')

    if use_exposure:
        try:
            exposure_table = datasources.get('exposure', 'bigalpha_2026_exposure')
            exp_select = ', '.join(exposure_features)
            exp_test_start = (
                pd.to_datetime(start_date) - pd.Timedelta(days=10)
            ).strftime('%Y-%m-%d %H:%M:%S')
            exp_test = dai.query(
                f"SELECT date, instrument, {exp_select} FROM {exposure_table}",
                filters={'date': [exp_test_start, end_date]}, compression=True,
            ).df()
            if not exp_test.empty:
                exp_test['date'] = pd.to_datetime(exp_test['date'])
                for c in exposure_features:
                    exp_test[c] = pd.to_numeric(exp_test[c], errors='coerce')
                test_df = pd.merge(test_df, exp_test, how='left', on=['date', 'instrument'])
                for c in exposure_features:
                    test_df[c] = test_df.groupby('instrument')[c].ffill()
        except Exception as e:
            logger.warning("测试期exposure失败", reason=str(e)[:100])

    for col in all_features:
        test_df[col] = test_df[col].fillna(train_medians.get(col, 0.0))

    test_df['factor'] = model.predict(test_df[all_features])

    # ============================================================
    # Step 3: 对齐成分股 + 后处理
    # ============================================================
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])

    test_df['date'] = pd.to_datetime(test_df['date'])
    result = pd.merge(
        test_df[['date', 'instrument', 'factor']], stk_pool,
        how='inner', on=['date', 'instrument'],
    )

    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=['factor']).reset_index(drop=True)

    f_mean = result['factor'].mean()
    f_std = result['factor'].std()
    if f_std > 0:
        result['factor'] = result['factor'].clip(f_mean - 5*f_std, f_mean + 5*f_std)

    logger.info("残差+订单流因子构建完成",
                rows=len(result),
                mean=round(result['factor'].mean(), 6),
                std=round(result['factor'].std(), 6))

    return result[['date', 'instrument', 'factor']]


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial',
    }
    start_date = '2024-01-01 00:00:00'
    end_date   = '2024-12-31 23:59:59'

    logger.info(f"计算残差+订单流因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    if factor_data.empty:
        logger.error("因子计算返回空结果！")
    else:
        logger.info(f"因子行数: {len(factor_data)}")
        logger.info(f"因子统计: mean={factor_data['factor'].mean():.6f}, "
                    f"std={factor_data['factor'].std():.6f}, "
                    f"coverage={factor_data['factor'].notna().mean():.2%}")

        factor_pool = dai.query(
            "SELECT * FROM bigalpha_2026_factorlib",
            filters={'date': [start_date, end_date]},
        ).df()

        result = M.bigalpha_eval._latest(
            factor_data=factor_data,
            factor_pool=factor_pool,
            process_pools=False,
            show=True,
        )